# Where It Shows Up In Ai

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dujing82-blip/usaaio_numpy_course/blob/main/notebooks/05_where_it_shows_up_in_ai.ipynb)

**English • CPU • synthetic data • predict shapes before coding**

Run Setup, then work down the page. Edit only YOUR CODE HERE. A blank answer prints FAIL; that is normal. Complete every component until the grader says ALL PASS.

In [ ]:
# @title Setup (run first)
!pip -q install git+https://github.com/dujing82-blip/usaaio_numpy_course.git
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from usaaio_check import grade, check_rules, show_shape, NO_LOOPS, PURE
from usaaio_check.exercise_tests import evaluate, regression_args, mlp_args
from usaaio_check.datasets import random_embeddings, blobs, titanic_like, iaio_kmeans_table
np.set_printoptions(precision=4, suppress=True)
exercise_results = {}


## Before you start

AI models use the same array operations in larger combinations. A linear layer changes the last feature axis. Gradient descent repeatedly updates parameters. Attention compares queries with keys, turns scores into probabilities, and mixes values. Image batches add a batch axis to channel/height/width.

Write a shape beside every intermediate array. The exercises are CPU-sized and use synthetic data; they teach operations, not production model performance.

### Exercise 5.1 — `My_Linear_NumPy` (2025 R1 P2.3–2.4) ★★★

[E1]

**Source:** 2025 R1 P2.3–2.4. Adapted exercise; source tags supplied with the course specification.

**Task:** class with `in_features`, `out_features`, `weight (out,in)`, `bias (out,)`, `random_seed=42`, `forward(x)` for any leading dims

**Shape contract:** x: (...,in); forward output: (...,out); weights: (out,in), bias: (out,)

**Rules:** My_Linear_NumPy.forward: no-loops; My_Linear_NumPy.__init__: no-loops

**Predict first:** Before running, write the input and output shapes for each function. Contract: x: (...,in); forward output: (...,out); weights: (out,in), bias: (out,).

**My prediction:** _Write your answer here before coding._

In [ ]:
class My_Linear_NumPy:
    def __init__(self,in_features,out_features,random_seed=42):
        '__init__: x: (...,in); forward output: (...,out); weights: (out,in), bias: (out,). See the exercise task for the required behavior.'
        self.in_features,self.out_features = in_features,out_features
        np.random.seed(random_seed)
        self.weight = np.random.randn(out_features,in_features)
        self.bias = np.random.randn(out_features)
    def forward(self,x):
        'forward: x: (...,in); forward output: (...,out); weights: (out,in), bias: (out,). See the exercise task for the required behavior.'
        return x@self.weight.T+self.bias


In [ ]:
# @title Grade 5.1 (run, do not edit)
exercise_results['5.1'] = evaluate(globals(), '5.1')

**Why it matters:** This operation is a building block used by the AI tasks tagged above.

<details><summary>Hint 1: operation</summary>

Multiply on the last axis; leading dimensions remain unchanged.

</details>

<details><summary>Hint 2: shape trick</summary>

Check which axes survive. A missing axis or an extra length-one axis changes broadcasting. Verify with a tiny hand-checkable example.

</details>

In [ ]:
if exercise_results.get('5.1',False):
    model = My_Linear_NumPy(3,2)
    show_shape(vector=model.forward(np.zeros(3)),batch=model.forward(np.zeros((10,3))),sequence=model.forward(np.zeros((10,20,3))),extra=model.forward(np.zeros((10,20,30,3))))

### Exercise 5.2 — Parameter counting (2025 R1 P2.6, 2024 IAIO Q4, 2026 Sample P12.3) ★★

[E1]

**Source:** 2025 R1 P2.6, 2024 IAIO Q4, 2026 Sample P12.3. Adapted exercise; source tags supplied with the course specification.

**Task:** `n_params_linear(in_,out,bias)`, `n_params_mlp(sizes)`, `n_params_conv(in_c,out_c,k,bias)`, `n_params_ffn(d,f)`

**Shape contract:** output of each parameter counter: one integer

**Rules:** Only the starter imports; loops allowed unless a subtask says otherwise.

**Predict first:** Before running, write the input and output shapes for each function. Contract: output of each parameter counter: one integer.

MLP layers include biases; FFN here is two linear layers without biases.

**My prediction:** _Write your answer here before coding._

In [ ]:
def n_params_linear(in_,out,bias=True):
    'n_params_linear: output of each parameter counter: one integer. See the exercise task for the required behavior.'
    return out*in_+(out if bias else 0)
def n_params_mlp(sizes):
    'n_params_mlp: output of each parameter counter: one integer. See the exercise task for the required behavior.'
    return sum(n_params_linear(a,b) for a,b in zip(sizes[:-1],sizes[1:]))
def n_params_conv(in_c,out_c,k,bias=True):
    'n_params_conv: output of each parameter counter: one integer. See the exercise task for the required behavior.'
    return out_c*in_c*k*k+(out_c if bias else 0)
def n_params_ffn(d,f):
    'n_params_ffn: output of each parameter counter: one integer. See the exercise task for the required behavior.'
    return 2*d*f


In [ ]:
# @title Grade 5.2 (run, do not edit)
exercise_results['5.2'] = evaluate(globals(), '5.2')

**Why it matters:** This operation is a building block used by the AI tasks tagged above.

<details><summary>Hint 1: operation</summary>

Count the weight entries, then add biases when present.

</details>

<details><summary>Hint 2: shape trick</summary>

Check which axes survive. A missing axis or an extra length-one axis changes broadcasting. Verify with a tiny hand-checkable example.

</details>

### Exercise 5.3 — Tied-weight symmetric model (2025 R1 P2.7–2.9) ★★

[E1] [C3]

**Source:** 2025 R1 P2.7–2.9. Adapted exercise; source tags supplied with the course specification.

**Task:** `symmetric_forward(W, x)` → `W.T @ W @ x` for `x (in,)` and batched `x (B,in)` via `x @ W.T @ W`

**Shape contract:** W: (out,in), x: (in,) or (B,in); output: same shape as x

**Rules:** symmetric_forward: no-loops

**Predict first:** Before running, write the input and output shapes for each function. Contract: W: (out,in), x: (in,) or (B,in); output: same shape as x.

**My prediction:** _Write your answer here before coding._

In [ ]:
def symmetric_forward(W,x):
    'symmetric_forward: W: (out,in), x: (in,) or (B,in); output: same shape as x. See the exercise task for the required behavior.'
    return x@W.T@W


In [ ]:
# @title Grade 5.3 (run, do not edit)
exercise_results['5.3'] = evaluate(globals(), '5.3')

**Why it matters:** This operation is a building block used by the AI tasks tagged above.

<details><summary>Hint 1: operation</summary>

For batches, samples are rows and both projections act on the right.

</details>

<details><summary>Hint 2: shape trick</summary>

Check which axes survive. A missing axis or an extra length-one axis changes broadcasting. Verify with a tiny hand-checkable example.

</details>

### Exercise 5.4 — Gradient descent for linear regression (E2) ★★★

[E2]

**Source:** Operation practice adapted from the supplied knowledge map. Adapted exercise; source tags supplied with the course specification.

**Task:** `fit_gd(X,y,lr,n_iter)` → `(theta, loss_history)` using `grad = -2/N * X.T @ (y - X@theta)`; only the iteration loop

**Shape contract:** X: (N,d), y: (N,); outputs: theta (d,), history (n_iter,)

**Rules:** fit_gd: at most 1 explicit iteration loop

**Predict first:** Before running, write the input and output shapes for each function. Contract: X: (N,d), y: (N,); outputs: theta (d,), history (n_iter,).

**My prediction:** _Write your answer here before coding._

In [ ]:
def fit_gd(X,y,lr,n_iter):
    'fit_gd: X: (N,d), y: (N,); outputs: theta (d,), history (n_iter,). See the exercise task for the required behavior.'
    theta = np.zeros(X.shape[1])
    history = np.zeros(n_iter)
    for i in range(n_iter):
        grad = -2/X.shape[0]*X.T@(y-X@theta)
        theta -= lr*grad
        history[i] = np.mean((y-X@theta)**2)
    return theta,history


In [ ]:
# @title Grade 5.4 (run, do not edit)
exercise_results['5.4'] = evaluate(globals(), '5.4')

**Why it matters:** This operation is a building block used by the AI tasks tagged above.

<details><summary>Hint 1: operation</summary>

Initialize theta to zeros; record the MSE after each update.

</details>

<details><summary>Hint 2: shape trick</summary>

Check which axes survive. A missing axis or an extra length-one axis changes broadcasting. Verify with a tiny hand-checkable example.

</details>

In [ ]:
# @title Plot after passing
if exercise_results.get('5.4', False):
    X,y,lr,n = regression_args(np.random.default_rng(2026)); theta,history = fit_gd(X,y,lr,n)
    plt.plot(history); plt.xlabel('Iteration'); plt.ylabel('MSE'); plt.show()


### Exercise 5.5 — `My_Log_Reg` (2025 R1 P3.16–3.18) ★★★

[E3]

**Source:** 2025 R1 P3.16–3.18. Adapted exercise; source tags supplied with the course specification.

**Task:** `my_sigmoid`, class with `solver∈{GD,Newton}`, `lr`, `num_iter`, `coef_ (d,)`, `fit` (one loop; `grad=X.T@(z-y)`, `H=X.T@np.diag(z*(1-z))@X`, Newton uses `np.linalg.inv(H)@grad`), `predict` with `np.where`, `score` = accuracy; raise `ValueError('Invalid solver')` otherwise

**Shape contract:** X: (N,d), y: (N,); coef_: (d,), predictions: (N,), score: scalar

**Rules:** My_Log_Reg.fit: at most 1 explicit iteration loop; My_Log_Reg.predict: no-loops; My_Log_Reg.score: no-loops

**Predict first:** Before running, write the input and output shapes for each function. Contract: X: (N,d), y: (N,); coef_: (d,), predictions: (N,), score: scalar.

The blobs overlap slightly and labels include a few flips to keep the Newton Hessian well-behaved.

**My prediction:** _Write your answer here before coding._

In [ ]:
def my_sigmoid(z):
    'my_sigmoid: X: (N,d), y: (N,); coef_: (d,), predictions: (N,), score: scalar. See the exercise task for the required behavior.'
    return 1/(1+np.exp(-np.clip(z,-500,500)))
class My_Log_Reg:
    def __init__(self,solver='GD',lr=0.01,num_iter=20):
        '__init__: X: (N,d), y: (N,); coef_: (d,), predictions: (N,), score: scalar. See the exercise task for the required behavior.'
        if solver not in ('GD','Newton'):
            raise ValueError('Invalid solver')
        self.solver,self.lr,self.num_iter = solver,lr,num_iter
        self.coef_ = None
    def fit(self,X,y):
        'fit: X: (N,d), y: (N,); coef_: (d,), predictions: (N,), score: scalar. See the exercise task for the required behavior.'
        self.coef_ = np.zeros(X.shape[1])
        for _ in range(self.num_iter):
            z = my_sigmoid(X@self.coef_)
            grad = X.T@(z-y)
            if self.solver=='Newton':
                H = X.T@np.diag(z*(1-z))@X
                self.coef_ -= np.linalg.inv(H)@grad
            else:
                self.coef_ -= self.lr*grad
        return self
    def predict(self,X):
        'predict: X: (N,d), y: (N,); coef_: (d,), predictions: (N,), score: scalar. See the exercise task for the required behavior.'
        return np.where(my_sigmoid(X@self.coef_)>=0.5,1,0)
    def score(self,X,y):
        'score: X: (N,d), y: (N,); coef_: (d,), predictions: (N,), score: scalar. See the exercise task for the required behavior.'
        return np.mean(self.predict(X)==y)


In [ ]:
# @title Grade 5.5 (run, do not edit)
exercise_results['5.5'] = evaluate(globals(), '5.5')

**Why it matters:** This operation is a building block used by the AI tasks tagged above.

<details><summary>Hint 1: operation</summary>

Use X.T @ (z-y); choose a GD or Newton update inside one iteration loop.

</details>

<details><summary>Hint 2: shape trick</summary>

Check which axes survive. A missing axis or an extra length-one axis changes broadcasting. Verify with a tiny hand-checkable example.

</details>

### Exercise 5.6 — Attention in NumPy (2025 R2 P2.5) ★★★

[E4]

**Source:** 2025 R2 P2.5. Adapted exercise; source tags supplied with the course specification.

**Task:** `mha_numpy(x, y, WQ, WK, WV, WO, H)` with weights given as `(H·d, D)` matrices: project, `reshape(B,L,H,d)`, `transpose(0,2,1,3)`, scores, softmax, `@V`, merge, out-projection; return `(B,L1,D1)`; **a shape comment after every line is required** (grader checks only values; the markdown asks for comments)

**Shape contract:** x: (B,L1,D), y: (B,L2,D); output: (B,L1,D1)

**Rules:** mha_numpy: no-loops

**Predict first:** Which axis does softmax normalize: queries or keys?

Write a shape comment after every line.

**My prediction:** _Write your answer here before coding._

In [ ]:
def mha_numpy(x,y,WQ,WK,WV,WO,H):
    'mha_numpy: x: (B,L1,D), y: (B,L2,D); output: (B,L1,D1). See the exercise task for the required behavior.'
    B,L1,D = x.shape  # x: (B,L1,D)
    L2 = y.shape[1]  # y: (B,L2,D)
    d = WQ.shape[0]//H  # each head: d
    Q = (x@WQ.T).reshape(B,L1,H,d).transpose(0,2,1,3)  # (B,H,L1,d)
    K = (y@WK.T).reshape(B,L2,H,d).transpose(0,2,1,3)  # (B,H,L2,d)
    V = (y@WV.T).reshape(B,L2,H,d).transpose(0,2,1,3)  # (B,H,L2,d)
    scores = Q@K.transpose(0,1,3,2)/np.sqrt(d)  # (B,H,L1,L2)
    e = np.exp(scores-scores.max(-1,keepdims=True))  # (B,H,L1,L2)
    probs = e/e.sum(-1,keepdims=True)  # (B,H,L1,L2)
    merged = (probs@V).transpose(0,2,1,3).reshape(B,L1,H*d)  # (B,L1,H*d)
    return merged@WO.T  # (B,L1,D1)


In [ ]:
# @title Grade 5.6 (run, do not edit)
exercise_results['5.6'] = evaluate(globals(), '5.6')

**Why it matters:** This operation is a building block used by the AI tasks tagged above.

<details><summary>Hint 1: operation</summary>

Split heads, compute scores, normalize keys, multiply V, and merge.

</details>

<details><summary>Hint 2: shape trick</summary>

Check which axes survive. A missing axis or an extra length-one axis changes broadcasting. Verify with a tiny hand-checkable example.

</details>

### Exercise 5.7 — GQA by broadcasting (2025 R2 P2.7) ★★★

[E4]

**Source:** 2025 R2 P2.7. Adapted exercise; source tags supplied with the course specification.

**Task:** `gqa_numpy(..., H, G)` with `K,V` having `G` heads; `Q.reshape(B,L1,H//G,G,d)`, `K.reshape(B,L2,1,G,d)` so the size-1 axis broadcasts

**Shape contract:** x: (B,L1,D), y: (B,L2,D); output: (B,L1,D1)

**Rules:** gqa_numpy: no-loops; gqa_numpy: no-kv-copies

**Predict first:** Before running, write the input and output shapes for each function. Contract: x: (B,L1,D), y: (B,L2,D); output: (B,L1,D1).

H is divisible by G; the head order is (repeat index, group index).

**My prediction:** _Write your answer here before coding._

In [ ]:
def gqa_numpy(x,y,WQ,WK,WV,WO,H,G):
    'gqa_numpy: x: (B,L1,D), y: (B,L2,D); output: (B,L1,D1). See the exercise task for the required behavior.'
    B,L1,D = x.shape
    L2 = y.shape[1]
    d = WQ.shape[0]//H
    Q = (x@WQ.T).reshape(B,L1,H//G,G,d).transpose(0,2,3,1,4)
    K = (y@WK.T).reshape(B,L2,1,G,d).transpose(0,2,3,1,4)
    V = (y@WV.T).reshape(B,L2,1,G,d).transpose(0,2,3,1,4)
    scores = Q@K.swapaxes(-2,-1)/np.sqrt(d)
    e = np.exp(scores-scores.max(-1,keepdims=True))
    probs = e/e.sum(-1,keepdims=True)
    merged = (probs@V).transpose(0,3,1,2,4).reshape(B,L1,H*d)
    return merged@WO.T


In [ ]:
# @title Grade 5.7 (run, do not edit)
exercise_results['5.7'] = evaluate(globals(), '5.7')

**Why it matters:** This operation is a building block used by the AI tasks tagged above.

<details><summary>Hint 1: operation</summary>

Insert a size-one group-repeat axis for K and V instead of copying.

</details>

<details><summary>Hint 2: shape trick</summary>

Check which axes survive. A missing axis or an extra length-one axis changes broadcasting. Verify with a tiny hand-checkable example.

</details>

### Exercise 5.8 — Nearest-token dictionary (2026 R1 P5.9) ★★★

[E5]

**Source:** 2026 R1 P5.9. Adapted exercise; source tags supplied with the course specification.

**Task:** `nearest_dict(tokens, W)` → `{token: most similar other token}` using `S=W@W.T`, `fill_diagonal(-inf)`, `argmax(axis=1)`

**Shape contract:** N tokens and W: (N,d); output: dictionary of N nearest-token names

**Rules:** nearest_dict: final-dict-only

**Predict first:** Before running, write the input and output shapes for each function. Contract: N tokens and W: (N,d); output: dictionary of N nearest-token names.

One final dict comprehension is allowed; rows are unit vectors.

**My prediction:** _Write your answer here before coding._

In [ ]:
def nearest_dict(tokens,W):
    'nearest_dict: N tokens and W: (N,d); output: dictionary of N nearest-token names. See the exercise task for the required behavior.'
    S = W@W.T
    np.fill_diagonal(S,-np.inf)
    indices = np.argmax(S,axis=1)
    return {token:tokens[j] for token,j in zip(tokens,indices)}


In [ ]:
# @title Grade 5.8 (run, do not edit)
exercise_results['5.8'] = evaluate(globals(), '5.8')

**Why it matters:** This operation is a building block used by the AI tasks tagged above.

<details><summary>Hint 1: operation</summary>

Exclude the diagonal before argmax; build the dictionary only at the end.

</details>

<details><summary>Hint 2: shape trick</summary>

Check which axes survive. A missing axis or an extra length-one axis changes broadcasting. Verify with a tiny hand-checkable example.

</details>

### Exercise 5.9 — Embedding compression end to end (2026 R1 P5, condensed) ★★★

[D4] [D5] [E5]

**Source:** 2026 R1 P5, condensed. Adapted exercise; source tags supplied with the course specification.

**Task:** given `vectors (N,100)`: unit rows → `S` → `U,s,Vt` → `Q,lam` → `‖S‖_F²` → error curve → plot

**Shape contract:** vectors: (N,d), N>=d; outputs: W (N,d), S (N,N), U (N,d), s (d,), Vt (d,d), Q (N,d), lam (d,), norm scalar, curve (d,)

**Rules:** compress_embeddings: no-loops; compress_embeddings: no-matmul

**Predict first:** Before running, write the input and output shapes for each function. Contract: vectors: (N,d), N>=d; outputs: W (N,d), S (N,N), U (N,d), s (d,), Vt (d,d), Q (N,d), lam (d,), norm scalar, curve (d,).

Do not use linalg for row normalization; SVD is allowed for the later factorization.

**My prediction:** _Write your answer here before coding._

In [ ]:
def compress_embeddings(vectors):
    'compress_embeddings: vectors: (N,d), N>=d; outputs: W (N,d), S (N,N), U (N,d), s (d,), Vt (d,d), Q (N,d), lam (d,), norm scalar, curve (d,). See the exercise task for the required behavior.'
    W = vectors/np.sqrt(np.sum(vectors**2,axis=1,keepdims=True))
    S = np.sum(W[:,None,:]*W[None,:,:],axis=-1)
    U,s,Vt = np.linalg.svd(W,full_matrices=False)
    Q,lam = U,s**2
    norm_sq = np.sum(s**4)
    curve = 1-np.cumsum(s**4)/norm_sq
    return W,S,U,s,Vt,Q,lam,norm_sq,curve


In [ ]:
# @title Grade 5.9 (run, do not edit)
exercise_results['5.9'] = evaluate(globals(), '5.9')

**Why it matters:** This operation is a building block used by the AI tasks tagged above.

<details><summary>Hint 1: operation</summary>

Normalize, compute all similarities, then use SVD and cumulative fourth powers.

</details>

<details><summary>Hint 2: shape trick</summary>

Check which axes survive. A missing axis or an extra length-one axis changes broadcasting. Verify with a tiny hand-checkable example.

</details>

In [ ]:
# @title Plot after passing
if exercise_results.get('5.9', False):
    parts = compress_embeddings(random_embeddings(N=60,d=20,seed=2026)[1]); curve = parts[-1]
    plt.plot(np.arange(1,len(curve)+1),curve); plt.xlabel('Rank r'); plt.ylabel('Relative squared error of S'); plt.show()


### Exercise 5.10 — Image batch (2025 R2 P3.4) ★★

[E6]

**Source:** 2025 R2 P3.4. Adapted exercise; source tags supplied with the course specification.

**Task:** `to_batch(list_of_HWC_uint8)` → float array `(B,3,224,224)` in `[-1,1]` via `np.transpose(img,(2,0,1))/255*2-1` and `np.stack`; grader checks range, shape and one pixel

**Shape contract:** input: list of B uint8 (224,224,3) images; output: floating (B,3,224,224)

**Rules:** Only the starter imports; loops allowed unless a subtask says otherwise.

**Predict first:** Before running, write the input and output shapes for each function. Contract: input: list of B uint8 (224,224,3) images; output: floating (B,3,224,224).

A loop or comprehension over images is allowed.

**My prediction:** _Write your answer here before coding._

In [ ]:
def to_batch(images):
    'to_batch: input: list of B uint8 (224,224,3) images; output: floating (B,3,224,224). See the exercise task for the required behavior.'
    return np.stack([np.transpose(img,(2,0,1))/255*2-1 for img in images])


In [ ]:
# @title Grade 5.10 (run, do not edit)
exercise_results['5.10'] = evaluate(globals(), '5.10')

**Why it matters:** This operation is a building block used by the AI tasks tagged above.

<details><summary>Hint 1: operation</summary>

Transpose and scale each image, then stack the batch.

</details>

<details><summary>Hint 2: shape trick</summary>

Check which axes survive. A missing axis or an extra length-one axis changes broadcasting. Verify with a tiny hand-checkable example.

</details>

### Exercise 5.11 — Triangle classifier with threshold units (2026 Sample P11 / 2026 R1 P7.4) ★★

[B6] [E1]

**Source:** 2026 Sample P11 / 2026 R1 P7.4. Adapted exercise; source tags supplied with the course specification.

**Task:** `inside_triangle(P)` for `P (B,2)` → `(B,)` 0/1 using `W (3,2)`, `b (3,)`, step `(z>=0)`, AND via `np.prod(axis=1)`; vertices `(0,0),(1,0),(0,2)`; second case vertices `(0,0),(3,1),(1,5)` (student derives the three half-plane inequalities)

**Shape contract:** P: (B,2); output: integer labels (B,)

**Rules:** inside_triangle: no-loops

**Predict first:** Before running, write the input and output shapes for each function. Contract: P: (B,2); output: integer labels (B,).

Boundary points count as inside. variant=2 uses vertices (0,0), (3,1), (1,5).

**My prediction:** _Write your answer here before coding._

In [ ]:
def inside_triangle(P,variant=1):
    'inside_triangle: P: (B,2); output: integer labels (B,). See the exercise task for the required behavior.'
    if variant==1:
        W,b = np.array([[1.,0.],[0.,1.],[-2.,-1.]]),np.array([0.,0.,2.])
    else:
        W,b = np.array([[-1.,3.],[-4.,-2.],[5.,-1.]]),np.array([0.,14.,0.])
    return np.prod((P@W.T+b)>=0,axis=1).astype(int)


In [ ]:
# @title Grade 5.11 (run, do not edit)
exercise_results['5.11'] = evaluate(globals(), '5.11')

**Why it matters:** This operation is a building block used by the AI tasks tagged above.

<details><summary>Hint 1: operation</summary>

Derive an inward-pointing inequality for each edge; multiply boolean tests.

</details>

<details><summary>Hint 2: shape trick</summary>

Check which axes survive. A missing axis or an extra length-one axis changes broadcasting. Verify with a tiny hand-checkable example.

</details>

### Exercise 5.12 — Softmax temperature (2025 R2 P3.20) ★★

[B1] [E4]

**Source:** 2025 R2 P3.20. Adapted exercise; source tags supplied with the course specification.

**Task:** `softmax_T(z, tau)`; grader checks `tau→0` gives one-hot on the max, `tau→∞` gives uniform

**Shape contract:** z: (...,K), tau>0; output: (...,K)

**Rules:** softmax_T: no-loops

**Predict first:** Before running, write the input and output shapes for each function. Contract: z: (...,K), tau>0; output: (...,K).

**My prediction:** _Write your answer here before coding._

In [ ]:
def softmax_T(z,tau):
    'softmax_T: z: (...,K), tau>0; output: (...,K). See the exercise task for the required behavior.'
    scaled = z/tau
    e = np.exp(scaled-scaled.max(-1,keepdims=True))
    return e/e.sum(-1,keepdims=True)


In [ ]:
# @title Grade 5.12 (run, do not edit)
exercise_results['5.12'] = evaluate(globals(), '5.12')

**Why it matters:** This operation is a building block used by the AI tasks tagged above.

<details><summary>Hint 1: operation</summary>

Divide logits by positive temperature before stable softmax.

</details>

<details><summary>Hint 2: shape trick</summary>

Check which axes survive. A missing axis or an extra length-one axis changes broadcasting. Verify with a tiny hand-checkable example.

</details>

### Exercise 5.13 — CLIP-style contrastive loss in NumPy (2025 R2 P3.15) ★★★

[E4] [B5]

**Source:** 2025 R2 P3.15. Adapted exercise; source tags supplied with the course specification.

**Task:** `clip_loss(I, T, log_tau)`: unit-normalize rows, `sim = I @ T.T`, symmetric cross-entropy on the diagonal with `log_softmax` along axis 0 and axis 1

**Shape contract:** I,T: (B,d), log_tau scalar; output: scalar loss

**Rules:** clip_loss: no-loops

**Predict first:** Before running, write the input and output shapes for each function. Contract: I,T: (B,d), log_tau scalar; output: scalar loss.

Here log_tau is log(temperature), so divide by exp(log_tau). Some implementations use a log scale and multiply instead; these conventions have opposite signs.

**My prediction:** _Write your answer here before coding._

In [ ]:
def clip_loss(I,T,log_tau):
    'clip_loss: I,T: (B,d), log_tau scalar; output: scalar loss. See the exercise task for the required behavior.'
    I = I/np.linalg.norm(I,axis=1,keepdims=True)
    T = T/np.linalg.norm(T,axis=1,keepdims=True)
    logits = (I@T.T)/np.exp(log_tau)
    row_max = logits.max(1,keepdims=True)
    col_max = logits.max(0,keepdims=True)
    row_log = logits-row_max-np.log(np.exp(logits-row_max).sum(1,keepdims=True))
    col_log = logits-col_max-np.log(np.exp(logits-col_max).sum(0,keepdims=True))
    return -0.5*(np.diag(row_log).mean()+np.diag(col_log).mean())


In [ ]:
# @title Grade 5.13 (run, do not edit)
exercise_results['5.13'] = evaluate(globals(), '5.13')

**Why it matters:** This operation is a building block used by the AI tasks tagged above.

<details><summary>Hint 1: operation</summary>

Compute log-softmax for both directions and take the diagonal losses.

</details>

<details><summary>Hint 2: shape trick</summary>

Check which axes survive. A missing axis or an extra length-one axis changes broadcasting. Verify with a tiny hand-checkable example.

</details>

## Checklist
- [ ] I can explain and use B1 (see the README knowledge map).
- [ ] I can explain and use B5 (see the README knowledge map).
- [ ] I can explain and use B6 (see the README knowledge map).
- [ ] I can explain and use C3 (see the README knowledge map).
- [ ] I can explain and use D4 (see the README knowledge map).
- [ ] I can explain and use D5 (see the README knowledge map).
- [ ] I can explain and use E1 (see the README knowledge map).
- [ ] I can explain and use E2 (see the README knowledge map).
- [ ] I can explain and use E3 (see the README knowledge map).
- [ ] I can explain and use E4 (see the README knowledge map).
- [ ] I can explain and use E5 (see the README knowledge map).
- [ ] I can explain and use E6 (see the README knowledge map).
- [ ] Every exercise says ALL PASS.
- [ ] I reviewed my shape predictions and can explain my mistakes.

Next: [07 pytorch bridge](07_pytorch_bridge.ipynb).